# Episode 9: Turning Text into Training Examples
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2009%20-%20Training%20Examples.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 9, one step per section. It uses only `numpy` and the standard library (already installed in Colab), so choose **Runtime, Run all** and nothing else is needed.

Covers: the counting convention (n tokens hold n − 1 terms), the nine (context, target) terms of the running sentence, the window cut at n_max = 3, stride-1 windows, mini-batch sizes, the identity total surprisal = N · L, a window-leakage toy, and the check question. The nine probabilities are illustrative (only 0.30 for "mat" is Episode 1's), and the tokens are word-level toys.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

In [1]:
import numpy as np
from fractions import Fraction
from math import exp, log
print("numpy", np.__version__)

numpy 2.5.3


## Step 1: the pairs
No start token: the first token has no context, so n = 10 tokens hold n − 1 = 9 terms, and N counts the terms summed in L.

In [2]:
tokens = "The cat sat on the mat because it was tired".split()      # x1 ... x10, word-level toys
n = len(tokens)
p = {2: 0.10, 3: 0.25, 4: 0.50, 5: 0.40, 6: 0.30, 7: 0.10, 8: 0.20, 9: 0.50, 10: 0.10}   # p(x_t | x_<t), illustrative
N = len(p)
assert n == 10 and N == n - 1 == 9

ell = {t: -np.log(p[t]) for t in p}
print(" t  context                 target     p     ell (nats)")
for t in range(2, n + 1):
    print(f"{t:2d}  {' '.join(tokens[:t-1]):22s}  {tokens[t-1]:8s} {p[t]:.2f}   {ell[t]:.3f}")

expected = {2: 2.303, 3: 1.386, 4: 0.693, 5: 0.916, 6: 1.204, 7: 2.303, 8: 1.609, 9: 0.693, 10: 2.303}
assert {t: round(v, 3) for t, v in ell.items()} == expected
assert p[6] == 0.30 and round(ell[6], 3) == 1.204        # the one number that is Episode 1's

 t  context                 target     p     ell (nats)
 2  The                     cat      0.10   2.303
 3  The cat                 sat      0.25   1.386
 4  The cat sat             on       0.50   0.693
 5  The cat sat on          the      0.40   0.916
 6  The cat sat on the      mat      0.30   1.204
 7  The cat sat on the mat  because  0.10   2.303
 8  The cat sat on the mat because  it       0.20   1.609
 9  The cat sat on the mat because it  was      0.50   0.693
10  The cat sat on the mat because it was  tired    0.10   2.303


## Step 2: the window cut and stride-1 windows
Cap the context at n_max = 3. Then slide a 4-token window one token at a time (n_max + 1 tokens: 3 inputs, 3 targets).

In [3]:
n_max = 3
def cut_context(t):                      # tokens x_{max(1, t-n_max)} ... x_{t-1}, 1-based t
    return tokens[max(0, t - 1 - n_max): t - 1]

assert " ".join(cut_context(6)) == "sat on the"            # not "The cat sat on the"
assert " ".join(cut_context(10)) == "because it was"        # "cat" is out of view when "tired" is predicted
print("t = 6 reads :", " ".join(cut_context(6)))
print("t = 10 reads:", " ".join(cut_context(10)))

win = n_max + 1
windows = [list(range(s, s + win)) for s in range(1, n - win + 2)]   # 1-based token positions
print("windows:", len(windows))
terms = [(w[0], w[1:]) for w in windows]
per_window_terms = n_max                                   # each window has 3 (context, target) terms
total_terms = len(windows) * per_window_terms
assert len(windows) == 7 and total_terms == 21

# how many times each target position is scored across the windows
from collections import Counter
count = Counter()
for w in windows:
    for pos in w[1:]:
        count[pos] += 1
print("times each target is scored:", dict(sorted(count.items())))
assert sorted(count) == list(range(2, 11)) and len(count) == 9          # the same 9 targets
assert all(count[pos] == 3 for pos in range(4, 9))                      # positions 4 to 8: three windows each

M_prime = 2
N_prime = M_prime * n_max
assert N_prime == 6
print("mini-batch of M' = 2 windows holds N' =", N_prime, "terms")

# dependence toy: loss values 0, 0, 6, 6
vals = np.array([0.0, 0.0, 6.0, 6.0])
var_independent_mean_of_two = vals.var() / 2                     # two independent draws from the four values
pair_means = np.array([vals[i:i + 2].mean() for i in range(len(vals) - 1)])   # two consecutive terms
print("variance of the mean of two independent draws:", var_independent_mean_of_two)
print("variance of the mean of two consecutive terms:", pair_means.var())
assert var_independent_mean_of_two == 4.5 and pair_means.var() == 6.0

t = 6 reads : sat on the
t = 10 reads: because it was
windows: 7
times each target is scored: {2: 1, 3: 2, 4: 3, 5: 3, 6: 3, 7: 3, 8: 3, 9: 2, 10: 1}
mini-batch of M' = 2 windows holds N' = 6 terms
variance of the mean of two independent draws: 4.5
variance of the mean of two consecutive terms: 6.0


## Step 3: the loss over the sequence

In [4]:
total = sum(ell.values())
L = total / N
PP = exp(L)
print(f"sum = {total:.3f} nats, L = {L:.3f}, PP = {PP:.2f}")
assert round(total, 3) == 13.410 and round(L, 3) == 1.490 and round(PP, 2) == 4.44

sum = 13.410 nats, L = 1.490, PP = 4.44


## Step 4: total surprisal equals N times the mean
The chain rule of probability (not the calculus chain rule): the product of the nine probabilities, then take −ln.

In [5]:
fr = [Fraction(1, 10), Fraction(1, 4), Fraction(1, 2), Fraction(2, 5), Fraction(3, 10),
      Fraction(1, 10), Fraction(1, 5), Fraction(1, 2), Fraction(1, 10)]
assert [float(f) for f in fr] == [p[t] for t in range(2, 11)]
prod = Fraction(1, 1)
for f in fr:
    prod *= f
print("product =", prod)
assert prod == Fraction(3, 2_000_000)

neg_ln = -log(float(prod))
assert abs(neg_ln - total) < 1e-9 and round(neg_ln, 3) == round(log(2_000_000 / 3), 3) == 13.410
assert abs(N * L - total) < 1e-12 and round(N * L, 3) == 13.410
pp_from_product = float(prod) ** (-1 / N)                    # (product)^(-1/N)
assert abs(pp_from_product - PP) < 1e-9 and round(pp_from_product, 2) == 4.44

# same total, different N: a tokenizer that cut the sentence into 14 pieces (N = 13), hypothetical
N14 = 13
L14 = total / N14
print(f"hypothetical 14 pieces: N = {N14}, L = {L14:.3f}, PP = {exp(L14):.2f}")
assert round(L14, 3) == 1.032 and round(exp(L14), 2) == 2.81

product = 3/2000000
hypothetical 14 pieces: N = 13, L = 1.032, PP = 2.81


## Step 5: window leakage (toy)
The ten tokens as a stand-in stream. Make the window x₄ … x₇ the validation window and see which other windows score the same targets.

In [6]:
# term = (window start s, target position, context positions); window s covers x_s ... x_{s+3}
def window_terms(s):
    return [(s, pos, tuple(range(s, pos))) for pos in range(s + 1, s + win)]

val_start = 4
val_terms = window_terms(val_start)
print("validation terms:", [(" ".join(tokens[c - 1] for c in ctx), "->", tokens[pos - 1]) for _, pos, ctx in val_terms])
assert [(" ".join(tokens[c - 1] for c in ctx), tokens[pos - 1]) for _, pos, ctx in val_terms] == [
    ("on", "the"), ("on the", "mat"), ("on the mat", "because")]

others = [s for s in range(1, 8) if s != val_start]
assert len(others) == 6
val_targets = {pos for _, pos, _ in val_terms}
twins = [(s, pos, ctx) for s in others for (_, pos, ctx) in window_terms(s) if pos in val_targets]
windows_with_twins = sorted({s for s, _, _ in twins})
for s, pos, ctx in twins:
    print("twin:", " ".join(tokens[c - 1] for c in ctx), "->", tokens[pos - 1])
print("windows with twin terms:", windows_with_twins, "; twin terms:", len(twins))
assert len(windows_with_twins) == 4 and len(twins) == 6
assert all(sum(1 for _, pos, _ in twins if pos == t) == 2 for t in val_targets)     # twice each

# non-overlapping spans: train on x1..x4 and x9 x10, validate on x5..x8
train_spans = [set(range(1, 5)), set(range(9, 11))]
val_span = set(range(5, 9))
shared = val_span & set().union(*train_spans)
train_targets = {pos for sp in train_spans for pos in sorted(sp)[1:]}      # first token of a span has no context
val_targets_new = set(sorted(val_span)[1:])
all_targets = set(range(2, 11))
lost = all_targets - train_targets - val_targets_new
print("shared tokens:", len(shared), "; targets lost at the cuts:", sorted(lost))
assert len(shared) == 0 and len(lost) == 2 and not (train_targets & val_targets_new)

validation terms: [('on', '->', 'the'), ('on the', '->', 'mat'), ('on the mat', '->', 'because')]
twin: cat sat on -> the
twin: sat on -> the
twin: sat on the -> mat
twin: the -> mat
twin: the mat -> because
twin: mat -> because
windows with twin terms: [2, 3, 5, 6] ; twin terms: 6
shared tokens: 0 ; targets lost at the cuts: [5, 9]


## GPT-1's mini-batch and the check question

In [7]:
# GPT-1: minibatches of 64 randomly sampled, contiguous sequences of 512 tokens
assert 64 * 512 == 32_768 and 64 * 511 == 32_704          # N' depends on whether the first token counts

# Check question: a 6-token sentence read whole, no start token. Probabilities 1/2, 1/4, 1/4, 1/4, 1/8.
probs = [Fraction(1, 2), Fraction(1, 4), Fraction(1, 4), Fraction(1, 4), Fraction(1, 8)]
N_a = len(probs)
tot_b = sum(-log(float(q)) for q in probs)
L_b = tot_b / N_a
print(f"(a) N = {N_a}")
print(f"(b) total = {tot_b:.3f} nats, L = {L_b:.3f}, PP = {exp(L_b):.2f}")
assert N_a == 5 and abs(tot_b - 10 * log(2)) < 1e-12 and round(tot_b, 3) == 6.931
assert abs(L_b - log(4)) < 1e-12 and round(L_b, 3) == 1.386 and round(exp(L_b), 6) == 4.0

# (c) with a start token and p = 1/8 for x1
tot_c = tot_b - log(1 / 8)
N_c = 6
L_c = tot_c / N_c
print(f"(c) N = {N_c}, total = {tot_c:.3f}, L = {L_c:.3f}, PP = {exp(L_c):.2f}")
assert abs(tot_c - 13 * log(2)) < 1e-12 and round(tot_c, 3) == 9.011
assert round(L_c, 3) == 1.502 and round(exp(L_c), 2) == round(2 ** (13 / 6), 2) == 4.49
print("All Episode 9 checks passed.")

(a) N = 5
(b) total = 6.931 nats, L = 1.386, PP = 4.00
(c) N = 6, total = 9.011, L = 1.502, PP = 4.49
All Episode 9 checks passed.
